# 1. Problem Statement & Goals 🎯
___
## Problem Statement
Ebuss, a growing e-commerce company with a significant market share in categories like household essentials, personal care, and electronics, aims to scale rapidly and compete with market leaders like Amazon and Flipkart.

To achieve this, Ebuss needs to leverage its vast data on user reviews and ratings. As a Senior Machine Learning Engineer, the core challenge is to build a sentiment-based product recommendation system. This system must not only recommend products based on user behaviors (ratings) but also refine those recommendations by analyzing the sentiment of the textual reviews associated with those products. The ultimate objective is to enhance the user experience by suggesting products that users are most likely to purchase and feel positive about.

## Goals
The project is divided into four main objectives to achieve the problem statement:

### 1. Data Sourcing and Sentiment Analysis

#### Objective: 
* Build a Machine Learning model to classify user reviews as Positive or Negative.

#### Key Tasks:

* Perform Exploratory Data Analysis (EDA), data cleaning, and text preprocessing.

* Extract features using techniques like Bag-of-Words, TF-IDF, or Word Embeddings.

* Train and evaluate at least three of the following classification models: Logistic Regression, Random Forest, XGBoost, or Naive Bayes.

* Select the best-performing model to predict user sentiment.

### 2. Building a Recommendation System

#### Objective: 
* Identify the most effective recommendation technique for the dataset.

#### Key Tasks:

* Develop both User-based and Item-based collaborative filtering recommendation systems.

* Analyze and compare their performance to select the best-suited system.

* Generate an initial list of 20 recommended products for a specific user based on their historical ratings.

### 3. Improving Recommendations using Sentiment Analysis

#### Objective: 
* Create a hybrid "Sentiment-Based Recommendation System."

#### Key Tasks:

* Integrate the chosen Sentiment Analysis model with the Recommendation System.

* Take the top 20 products recommended by the collaborative filtering system.

* Filter and rank these products based on their predicted sentiment scores.

* Output the final top 5 products that have the highest positive sentiment.

### 4. Deployment

#### Objective: 
* Make the solution accessible via a web interface.

#### Key Tasks:

* Build a web application using the Flask framework.

* Create a User Interface (UI) that accepts a username and displays the top 5 recommended products.

* Deploy the end-to-end application (Model + API + UI) on a cloud platform like Heroku.m

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pickle
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os

# Notebook Setup
from notebook_setup import NotebookInitializer
# Pass the path of the current file to the initializer
initializer = NotebookInitializer(Path(os.getcwd()).resolve())
initializer.setup_environment()

In [ ]:
# Local Utils
from utils import DataFileManager

# 1. DATA PREPARATION

In [ ]:
df = DataFileManager.load_csv_data(initializer.processed_data_dir / "df_final.csv")

In [ ]:
print("Original Columns:", df.columns)

In [ ]:
# Keep only required columns
df = df[['reviews_username', 'product_name', 'reviews_rating']].copy()

print(f"\nDataset Shape: {df.shape}")
print(f"Number of users: {df['reviews_username'].nunique()}")
print(f"Number of products: {df['product_name'].nunique()}")
print(f"Number of ratings: {len(df)}")
# Check for missing values
print(f"\nMissing values:")
print(df.isnull().sum())

# Remove missing values
df = df.dropna()

print(f"\nAfter removing missing values:")
print(f"Number of ratings: {len(df)}")

# Data statistics
print(f"\nRating statistics:")
print(df['reviews_rating'].describe())

# Sparsity calculation
n_users = df['reviews_username'].nunique()
n_products = df['product_name'].nunique()
n_ratings = len(df)
sparsity = 1 - (n_ratings / (n_users * n_products))
print(f"\nData sparsity: {sparsity:.2%}")
print(f"(Lower is better - means more user-product interactions)")

In [ ]:
# Define Minimum Thresholds (Adjust these values based on your data size!)
MIN_USER_REVIEWS = 2  # Users must have rated at least 2 products
MIN_PRODUCT_RATINGS = 2 # Products must have been rated by at least 2 users

# --- Filter Users ---
# Count how many products each user has reviewed
review_counts_by_user = df['reviews_username'].value_counts()
# Get the list of users that meet the minimum threshold
active_users = review_counts_by_user[review_counts_by_user >= MIN_USER_REVIEWS].index
# Filter the original DataFrame
df_filtered_users = df[df['reviews_username'].isin(active_users)]


# --- Filter Products ---
# Count how many users have rated each product
review_counts_by_product = df_filtered_users['product_name'].value_counts()
# Get the list of products that meet the minimum threshold
active_products = review_counts_by_product[review_counts_by_product >= MIN_PRODUCT_RATINGS].index
# Filter the DataFrame again
df_filtered_final = df_filtered_users[df_filtered_users['product_name'].isin(active_products)]


print(f"\nFiltered Dataset Shape: {df_filtered_final.shape}")
print(f"Lost users percentage: {((df['reviews_username'].nunique() - df_filtered_final['reviews_username'].nunique()) / df['reviews_username'].nunique()) * 100}")
print(f"Filtered Number of users: {df_filtered_final['reviews_username'].nunique()}")
print(f"Lost products percentage: {((df['product_name'].nunique() - df_filtered_final['product_name'].nunique()) / df['product_name'].nunique()) * 100}")
print(f"Filtered Number of products: {df_filtered_final['product_name'].nunique()}")
print(f"Filtered Number of ratings: {len(df_filtered_final)}")

# Data statistics
print(f"\nRating statistics:")
print(df_filtered_final['reviews_rating'].describe())

# Sparsity calculation
n_users = df_filtered_final['reviews_username'].nunique()
n_products = df_filtered_final['product_name'].nunique()
n_ratings = len(df_filtered_final)
sparsity = 1 - (n_ratings / (n_users * n_products))
print(f"\nFiltered Data sparsity: {sparsity:.2%}")
print(f"(Lower is better - means more user-product interactions)")

# 2. CREATE USER-ITEM MATRIX

In [ ]:
print("\n" + "="*80)
print("CREATING USER-ITEM RATING MATRIX")
print("="*80)

# Create pivot table: rows=users, columns=products, values=ratings
user_item_matrix = df_filtered_final.pivot_table(
    index='reviews_username',
    columns='product_name',
    values='reviews_rating',
    aggfunc='mean'  # If user rated same product multiple times, take average
)
print(f"\nUser-Item Matrix Shape: {user_item_matrix.shape}")
print(f"  Rows (Users): {user_item_matrix.shape[0]}")
print(f"  Columns (Products): {user_item_matrix.shape[1]}")
print(f"  Total cells: {user_item_matrix.shape[0] * user_item_matrix.shape[1]}")
print(f"  Filled cells: {user_item_matrix.notna().sum().sum()}")
print(f"  Empty cells: {user_item_matrix.isna().sum().sum()}")

# Show sample of the matrix
print("\nSample User-Item Matrix (first 5 users x 5 products):")
display(user_item_matrix.iloc[:5, :5])

# 3. TRAIN-TEST SPLIT

In [ ]:
print("\n" + "="*80)
print("TRAIN-TEST SPLIT")
print("="*80)

from sklearn.model_selection import train_test_split

# Split data: 80% train, 20% test
train_data, test_data = train_test_split(df_filtered_final, test_size=0.2, random_state=31)

print(f"Training set: {len(train_data)} ratings")
print(f"Test set: {len(test_data)} ratings")

# Create train user-item matrix
train_matrix = train_data.pivot_table(
    index='reviews_username',
    columns='product_name',
    values='reviews_rating',
    aggfunc='mean'
)

print(f"\nTraining Matrix Shape: {train_matrix.shape}")

# 4. USER-BASED COLLABORATIVE FILTERING

In [ ]:
def adjusted_cosine_similarity_users(user_item_matrix):
    """
    Calculate adjusted cosine similarity between users
    
    Adjusted cosine removes the user's average rating bias
    For each user, subtract their mean rating before computing similarity
    
    Parameters:
    -----------
    user_item_matrix : DataFrame
        User-item rating matrix (users as rows, items as columns)
    
    Returns:
    --------
    similarity_matrix : DataFrame
        User-user similarity matrix
    """
    print("\nCalculating user mean ratings...")
    # Calculate mean rating for each user (only on rated items)
    user_means = user_item_matrix.mean(axis=1)
    
    print("Subtracting user means (mean-centering)...")
    # Subtract user mean from each rating (mean-centering)
    # This removes user rating bias (some users rate higher/lower on average)
    user_item_centered = user_item_matrix.sub(user_means, axis=0)
    
    # Fill NaN with 0 for similarity calculation
    user_item_centered_filled = user_item_centered.fillna(0)
    
    print("Computing cosine similarity on centered data...")
    # Calculate cosine similarity on mean-centered data
    similarity = cosine_similarity(user_item_centered_filled)
    
    # Create DataFrame
    similarity_df = pd.DataFrame(
        similarity,
        index=user_item_matrix.index,
        columns=user_item_matrix.index
    )
    
    return similarity_df, user_means

In [ ]:
print("\n" + "="*80)
print("USER-BASED COLLABORATIVE FILTERING")
print("="*80)

user_similarity_adj, user_means = adjusted_cosine_similarity_users(train_matrix)

print(f"\nUser Similarity Matrix Shape: {user_similarity_adj.shape}")
print(f"\nSample Adjusted User Similarities (first 5x5):")
display(user_similarity_adj.iloc[:5, :5])

print(f"\nSample User Mean Ratings:")
print(user_means.head(10))

# 5. ADJUSTED COSINE SIMILARITY - ITEM-BASED

In [ ]:
def adjusted_cosine_similarity_items(user_item_matrix):
    """
    Calculate adjusted cosine similarity between items
    
    For item-based CF, we subtract each USER's mean rating
    This accounts for the fact that some users rate everything high/low
    
    Parameters:
    -----------
    user_item_matrix : DataFrame
        User-item rating matrix (users as rows, items as columns)
    
    Returns:
    --------
    similarity_matrix : DataFrame
        Item-item similarity matrix
    """
    print("\nCalculating user mean ratings...")
    # Calculate mean rating for each user
    user_means = user_item_matrix.mean(axis=1)
    
    print("Subtracting user means from ratings...")
    # Subtract each user's mean from their ratings
    user_item_centered = user_item_matrix.sub(user_means, axis=0)
    
    # Transpose: now items are rows
    item_user_centered = user_item_centered.T
    
    # Fill NaN with 0
    item_user_centered_filled = item_user_centered.fillna(0)
    
    print("Computing cosine similarity on centered data...")
    # Calculate cosine similarity between items
    similarity = cosine_similarity(item_user_centered_filled)
    
    # Create DataFrame
    similarity_df = pd.DataFrame(
        similarity,
        index=user_item_matrix.columns,
        columns=user_item_matrix.columns
    )
    
    return similarity_df

In [ ]:
print("\n" + "="*80)
print("CALCULATING ADJUSTED COSINE SIMILARITY - ITEM-BASED")
print("="*80)

# Calculate adjusted cosine similarity for items
item_similarity_adj = adjusted_cosine_similarity_items(train_matrix)

print(f"\nItem Similarity Matrix Shape: {item_similarity_adj.shape}")
print(f"\nSample Adjusted Item Similarities (first 5x5):")
display(item_similarity_adj.iloc[:5, :5])

# 6. USER-BASED CF WITH ADJUSTED COSINE

In [ ]:
def predict_user_based_adjusted(user, product, user_item_matrix, user_similarity_matrix, 
                                user_means, k=10):
    """
    Predict rating using user-based CF with adjusted cosine similarity
    
    Parameters:
    -----------
    user : str
        Username
    product : str
        Product name
    user_item_matrix : DataFrame
        User-item rating matrix
    user_similarity_matrix : DataFrame
        User-user adjusted cosine similarity matrix
    user_means : Series
        Mean rating for each user
    k : int
        Number of similar users to consider
    
    Returns:
    --------
    predicted_rating : float
        Predicted rating (1-5)
    """
    if user not in user_item_matrix.index:
        return user_item_matrix.mean().mean()
    
    if product not in user_item_matrix.columns:
        return user_means[user] if user in user_means.index else user_item_matrix.mean().mean()
    
    # Get similar users who rated this product
    similar_users = user_similarity_matrix[user].sort_values(ascending=False)
    similar_users = similar_users.drop(user, errors='ignore')
    
    # Get top-k similar users who rated this product
    rated_mask = user_item_matrix[product].notna()
    similar_users_rated = similar_users[rated_mask]
    top_k_users = similar_users_rated.head(k)
    
    if len(top_k_users) == 0:
        return user_means[user] if user in user_means.index else user_item_matrix.mean().mean()
    
    # Weighted average with user mean adjustment
    user_mean = user_means[user]
    
    # For each similar user: (their_rating - their_mean) * similarity
    # Then add back target user's mean
    numerator = 0
    denominator = 0
    
    for similar_user in top_k_users.index:
        similarity_score = top_k_users[similar_user]
        similar_user_rating = user_item_matrix.loc[similar_user, product]
        similar_user_mean = user_means[similar_user]
        
        # Add (rating - mean) * similarity
        numerator += similarity_score * (similar_user_rating - similar_user_mean)
        denominator += abs(similarity_score)
    
    if denominator == 0:
        return user_mean
    
    # Predicted rating = user_mean + weighted_average_of_deviations
    predicted_rating = user_mean + (numerator / denominator)
    
    # Clip to valid rating range
    predicted_rating = np.clip(predicted_rating, 1, 5)
    
    return predicted_rating

In [ ]:
def recommend_user_based_adjusted(user, user_item_matrix, user_similarity_matrix, 
                                 user_means, n=10, k=10):
    """
    Get top-N recommendations using user-based CF with adjusted cosine
    """
    if user not in user_item_matrix.index:
        product_avg_ratings = user_item_matrix.mean().sort_values(ascending=False)
        return pd.DataFrame({
            'product': product_avg_ratings.head(n).index,
            'predicted_rating': product_avg_ratings.head(n).values
        })
    
    # Get unrated products
    user_ratings = user_item_matrix.loc[user]
    unrated_products = user_ratings[user_ratings.isna()].index
    
    # Predict ratings
    predictions = []
    for product in unrated_products:
        pred_rating = predict_user_based_adjusted(
            user, product, user_item_matrix, user_similarity_matrix, user_means, k
        )
        predictions.append({'product': product, 'predicted_rating': pred_rating})
    
    # Sort by predicted rating
    recommendations = pd.DataFrame(predictions).sort_values('predicted_rating', ascending=False)
    
    return recommendations.head(n).reset_index(drop=True)

In [ ]:
print("\n" + "="*80)
print("USER-BASED CF WITH ADJUSTED COSINE SIMILARITY")
print("="*80)

print("\nEvaluating User-Based CF (Adjusted Cosine) on test set...")
user_based_predictions_adj = []

for idx, row in test_data.iterrows():
    user = row['reviews_username']
    product = row['product_name']
    actual_rating = row['reviews_rating']
    
    pred_rating = predict_user_based_adjusted(
        user, product, train_matrix, user_similarity_adj, user_means, k=10
    )
    user_based_predictions_adj.append({
        'user': user,
        'product': product,
        'actual': actual_rating,
        'predicted': pred_rating
    })

user_based_results_adj = pd.DataFrame(user_based_predictions_adj)

# Calculate metrics
user_based_rmse_adj = np.sqrt(mean_squared_error(
    user_based_results_adj['actual'], 
    user_based_results_adj['predicted']
))
user_based_mae_adj = mean_absolute_error(
    user_based_results_adj['actual'], 
    user_based_results_adj['predicted']
)

print(f"\n{'='*80}")
print("USER-BASED CF WITH ADJUSTED COSINE - PERFORMANCE")
print(f"{'='*80}")
print(f"RMSE: {user_based_rmse_adj:.4f}")
print(f"MAE:  {user_based_mae_adj:.4f}")

# 7. ITEM-BASED CF WITH ADJUSTED COSINE

In [ ]:
def predict_item_based_adjusted(user, product, user_item_matrix, item_similarity_matrix, 
                               user_means, k=10):
    """
    Predict rating using item-based CF with adjusted cosine similarity
    
    Parameters:
    -----------
    user : str
        Username
    product : str
        Product name
    user_item_matrix : DataFrame
        User-item rating matrix
    item_similarity_matrix : DataFrame
        Item-item adjusted cosine similarity matrix
    user_means : Series
        Mean rating for each user
    k : int
        Number of similar items to consider
    
    Returns:
    --------
    predicted_rating : float
        Predicted rating (1-5)
    """
    if user not in user_item_matrix.index:
        return user_item_matrix.mean().mean()
    
    if product not in user_item_matrix.columns:
        return user_means[user] if user in user_means.index else user_item_matrix.mean().mean()
    
    # Get similar items that the user has rated
    similar_items = item_similarity_matrix[product].sort_values(ascending=False)
    similar_items = similar_items.drop(product, errors='ignore')
    
    # Get top-k similar items rated by this user
    user_ratings = user_item_matrix.loc[user]
    rated_mask = user_ratings.notna()
    similar_items_rated = similar_items[rated_mask]
    top_k_items = similar_items_rated.head(k)
    
    if len(top_k_items) == 0:
        return user_means[user] if user in user_means.index else user_item_matrix.mean().mean()
    
    # Weighted average with adjustment
    user_mean = user_means[user]
    
    numerator = 0
    denominator = 0
    
    for similar_item in top_k_items.index:
        similarity_score = top_k_items[similar_item]
        user_rating_for_item = user_item_matrix.loc[user, similar_item]
        
        # Use the deviation from user's mean
        numerator += similarity_score * (user_rating_for_item - user_mean)
        denominator += abs(similarity_score)
    
    if denominator == 0:
        return user_mean
    
    # Predicted rating = user_mean + weighted_average_of_deviations
    predicted_rating = user_mean + (numerator / denominator)
    
    # Clip to valid rating range
    predicted_rating = np.clip(predicted_rating, 1, 5)
    
    return predicted_rating

In [ ]:
def recommend_item_based_adjusted(user, user_item_matrix, item_similarity_matrix, 
                                user_means, n=10, k=10):
    """
    Get top-N recommendations using item-based CF with adjusted cosine
    """
    if user not in user_item_matrix.index:
        product_avg_ratings = user_item_matrix.mean().sort_values(ascending=False)
        return pd.DataFrame({
            'product': product_avg_ratings.head(n).index,
            'predicted_rating': product_avg_ratings.head(n).values
        })
    
    # Get unrated products
    user_ratings = user_item_matrix.loc[user]
    unrated_products = user_ratings[user_ratings.isna()].index
    
    # Predict ratings
    predictions = []
    for product in unrated_products:
        pred_rating = predict_item_based_adjusted(
            user, product, user_item_matrix, item_similarity_matrix, user_means, k
        )
        predictions.append({'product': product, 'predicted_rating': pred_rating})
    
    # Sort by predicted rating
    recommendations = pd.DataFrame(predictions).sort_values('predicted_rating', ascending=False)
    
    return recommendations.head(n).reset_index(drop=True)

In [ ]:
print("\n" + "="*80)
print("ITEM-BASED CF WITH ADJUSTED COSINE SIMILARITY")
print("="*80)

# Evaluate
print("\nEvaluating Item-Based CF (Adjusted Cosine) on test set...")
item_based_predictions_adj = []

for idx, row in test_data.iterrows():
    user = row['reviews_username']
    product = row['product_name']
    actual_rating = row['reviews_rating']
    
    pred_rating = predict_item_based_adjusted(
        user, product, train_matrix, item_similarity_adj, user_means, k=10
    )
    item_based_predictions_adj.append({
        'user': user,
        'product': product,
        'actual': actual_rating,
        'predicted': pred_rating
    })

item_based_results_adj = pd.DataFrame(item_based_predictions_adj)

# Calculate metrics
item_based_rmse_adj = np.sqrt(mean_squared_error(
    item_based_results_adj['actual'], 
    item_based_results_adj['predicted']
))
item_based_mae_adj = mean_absolute_error(
    item_based_results_adj['actual'], 
    item_based_results_adj['predicted']
)

print(f"\n{'='*80}")
print("ITEM-BASED CF WITH ADJUSTED COSINE - PERFORMANCE")
print(f"{'='*80}")
print(f"RMSE: {item_based_rmse_adj:.4f}")
print(f"MAE:  {item_based_mae_adj:.4f}")

# 8. COMPARISON

In [ ]:
print("\n" + "="*80)
print("MODEL COMPARISON - ADJUSTED COSINE SIMILARITY")
print("="*80)

comparison = pd.DataFrame({
    'Model': ['User-Based CF (Adj Cosine)', 'Item-Based CF (Adj Cosine)'],
    'RMSE': [user_based_rmse_adj, item_based_rmse_adj],
    'MAE': [user_based_mae_adj, item_based_mae_adj]
})

print("\n" + comparison.to_string(index=False))

# Determine best model
if user_based_rmse_adj < item_based_rmse_adj:
    best_model = 'User-Based CF (Adjusted Cosine)'
    best_rmse = user_based_rmse_adj
    best_mae = user_based_mae_adj
    best_similarity_matrix = user_similarity_adj
    is_user_based = True
else:
    best_model = 'Item-Based CF (Adjusted Cosine)'
    best_rmse = item_based_rmse_adj
    best_mae = item_based_mae_adj
    best_similarity_matrix = item_similarity_adj
    is_user_based = False

print(f"\n🏆 BEST MODEL: {best_model}")
print(f"   RMSE: {best_rmse:.4f}")
print(f"   MAE:  {best_mae:.4f}")

print("\n" + "="*80)
print("WHY ADJUSTED COSINE IS BETTER")
print("="*80)
print("""
Regular Cosine Similarity:
  - Treats all ratings equally
  - Doesn't account for user rating bias
  - User who rates everything 5 looks similar to user who rates everything 3

Adjusted Cosine Similarity:
  ✓ Subtracts each user's mean rating
  ✓ Accounts for user rating bias (some users rate high, others low)
  ✓ Focuses on rating PATTERNS, not absolute values
  ✓ Generally produces MORE ACCURATE recommendations
  
Example:
  User A rates: [5, 5, 4] → loves everything
  User B rates: [3, 3, 2] → neutral on everything
  
  Regular cosine: High similarity (both consistent raters)
  Adjusted cosine: High similarity (both show same PATTERN of preferences)
  
  User A rates: [5, 5, 4]
  User C rates: [5, 2, 1]
  
  Regular cosine: Medium similarity (some overlap)
  Adjusted cosine: LOW similarity (different patterns after mean-centering)
""")

# 9. SAMPLE RECOMMENDATIONS

In [ ]:
print("\n" + "="*80)
print("SAMPLE RECOMMENDATIONS (ADJUSTED COSINE)")
print("="*80)

# Get 5 random users
sample_users = train_matrix.index[:5].tolist()

for user in sample_users:
    print(f"\n{'='*80}")
    print(f"Recommendations for User: {user}")
    print(f"User's average rating: {user_means[user]:.2f}" if user in user_means.index else "")
    print(f"{'='*80}")
    
    # User-based recommendations
    print("\n--- USER-BASED CF (Adjusted Cosine) ---")
    user_recs = recommend_user_based_adjusted(
        user, train_matrix, user_similarity_adj, user_means, n=5, k=10
    )
    if len(user_recs) > 0:
        for idx, row in user_recs.iterrows():
            print(f"{idx+1}. {row['product']:<50} (predicted: {row['predicted_rating']:.2f})")
    else:
        print("No recommendations available")
    
    # Item-based recommendations
    print("\n--- ITEM-BASED CF (Adjusted Cosine) ---")
    item_recs = recommend_item_based_adjusted(
        user, train_matrix, item_similarity_adj, user_means, n=5, k=10
    )
    if len(item_recs) > 0:
        for idx, row in item_recs.iterrows():
            print(f"{idx+1}. {row['product']:<50} (predicted: {row['predicted_rating']:.2f})")
    else:
        print("No recommendations available")

# 10. SAVE MODELS

In [ ]:
print("\n" + "="*80)
print("SAVING RECOMMENDATION SYSTEMS")
print("="*80)

import os
import pickle

os.makedirs('models/recommendation', exist_ok=True)

# Save user-item matrix
with open('models/recommendation/user_item_matrix.pkl', 'wb') as f:
    pickle.dump(train_matrix, f)
print("✓ User-item matrix saved")

# Save similarity matrices
with open('models/recommendation/user_similarity_matrix.pkl', 'wb') as f:
    pickle.dump(user_similarity_adj, f)
with open('models/recommendation/item_similarity_matrix.pkl', 'wb') as f:
    pickle.dump(item_similarity_adj, f)
print("✓ Similarity matrices saved")

# Save everything
recommendation_system = {
    'user_item_matrix': train_matrix,
    'user_similarity_matrix': user_similarity_adj,
    'item_similarity_matrix': item_similarity_adj,
    'user_means': user_means,
    'best_model': best_model,
    'is_user_based': is_user_based,
    'user_based_rmse': user_based_rmse_adj,
    'item_based_rmse': item_based_rmse_adj,
    'user_based_mae': user_based_mae_adj,
    'item_based_mae': item_based_mae_adj,
    'similarity_type': 'adjusted_cosine'
}

with open('models/recommendation/recommendation_system.pkl', 'wb') as f:
    pickle.dump(recommendation_system, f)
print("✓ Complete recommendation system saved")

In [ ]:
user_similarity, user_means = adjusted_cosine_similarity_users(user_item_matrix)

print(f"\nUser-User Similarity Matrix Shape: {user_similarity.shape}")
print(f"Similarity range: [{user_similarity.min().min():.4f}, {user_similarity.max().max():.4f}]")

# Show statistics
print("\nUser-User Similarity Statistics:")
# Get upper triangle (excluding diagonal)
mask = np.triu(np.ones_like(user_similarity), k=1).astype(bool)
similarities = user_similarity.where(mask).stack()
print(f"  Mean similarity: {similarities.mean():.4f}")
print(f"  Median similarity: {similarities.median():.4f}")
print(f"  Std deviation: {similarities.std():.4f}")

print("\nSample User-User Similarities (first 5x5):")
print(user_similarity.iloc[:5, :5])

# Find most similar user pairs
print("\nTop 10 Most Similar User Pairs:")
top_similarities = similarities.nlargest(10)
for idx, (users, sim) in enumerate(top_similarities.items(), 1):
    print(f"{idx:2d}. {users[0]:30s} <-> {users[1]:30s} : {sim:.4f}")

In [ ]:
item_similarity = adjusted_cosine_similarity_items(user_item_matrix)

print(f"\nItem-Item Similarity Matrix Shape: {item_similarity.shape}")
print(f"Similarity range: [{item_similarity.min().min():.4f}, {item_similarity.max().max():.4f}]")

# Show statistics
print("\nItem-Item Similarity Statistics:")
mask = np.triu(np.ones_like(item_similarity), k=1).astype(bool)
similarities_items = item_similarity.where(mask).stack()
print(f"  Mean similarity: {similarities_items.mean():.4f}")
print(f"  Median similarity: {similarities_items.median():.4f}")
print(f"  Std deviation: {similarities_items.std():.4f}")

print("\nSample Item-Item Similarities (first 5x5):")
print(item_similarity.iloc[:5, :5])

# Find most similar item pairs
print("\nTop 10 Most Similar Product Pairs:")
top_similarities_items = similarities_items.nlargest(10)
for idx, (items, sim) in enumerate(top_similarities_items.items(), 1):
    print(f"{idx:2d}. {items[0][:40]:40s} <-> {items[1][:40]:40s} : {sim:.4f}")

In [ ]:
print("\n" + "="*80)
print("EVALUATION VIA LEAVE-ONE-OUT CROSS-VALIDATION (SAMPLE)")
print("="*80)

# Sample 1000 random ratings for evaluation (to avoid long computation)
print("\nSampling 1000 random ratings for evaluation...")
sample_size = min(1000, len(df))
eval_sample = df.sample(n=sample_size, random_state=42)

print(f"Evaluating on {len(eval_sample)} ratings...")

# User-based evaluation
user_based_predictions = []
for idx, row in eval_sample.iterrows():
    user = row['reviews_username']
    product = row['product_name']
    actual_rating = row['reviews_rating']
    
    # Create temporary matrix without this rating
    temp_matrix = user_item_matrix.copy()
    if user in temp_matrix.index and product in temp_matrix.columns:
        temp_matrix.loc[user, product] = np.nan
        pred_rating = predict_user_based_adjusted(
            user, product, temp_matrix, user_similarity, user_means, k=10
        )
        user_based_predictions.append({
            'actual': actual_rating,
            'predicted': pred_rating
        })

user_based_results = pd.DataFrame(user_based_predictions)
user_based_rmse = np.sqrt(mean_squared_error(user_based_results['actual'], user_based_results['predicted']))
user_based_mae = mean_absolute_error(user_based_results['actual'], user_based_results['predicted'])

print(f"\nUSER-BASED CF (User-User) Performance:")
print(f"  RMSE: {user_based_rmse:.4f}")
print(f"  MAE:  {user_based_mae:.4f}")

# Item-based evaluation
item_based_predictions = []
for idx, row in eval_sample.iterrows():
    user = row['reviews_username']
    product = row['product_name']
    actual_rating = row['reviews_rating']
    
    # Create temporary matrix without this rating
    temp_matrix = user_item_matrix.copy()
    if user in temp_matrix.index and product in temp_matrix.columns:
        temp_matrix.loc[user, product] = np.nan
        
        pred_rating = predict_item_based_adjusted(
            user, product, temp_matrix, item_similarity, user_means, k=10
        )
        item_based_predictions.append({
            'actual': actual_rating,
            'predicted': pred_rating
        })

item_based_results = pd.DataFrame(item_based_predictions)
item_based_rmse = np.sqrt(mean_squared_error(item_based_results['actual'], item_based_results['predicted']))
item_based_mae = mean_absolute_error(item_based_results['actual'], item_based_results['predicted'])

print(f"\nITEM-BASED CF (Item-Item) Performance:")
print(f"  RMSE: {item_based_rmse:.4f}")
print(f"  MAE:  {item_based_mae:.4f}")

In [ ]:
print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)

comparison = pd.DataFrame({
    'Model': ['User-Based (User-User)', 'Item-Based (Item-Item)'],
    'RMSE': [user_based_rmse, item_based_rmse],
    'MAE': [user_based_mae, item_based_mae]
})

print("\n" + comparison.to_string(index=False))

if user_based_rmse < item_based_rmse:
    best_model = 'User-Based (User-User)'
    best_rmse = user_based_rmse
else:
    best_model = 'Item-Based (Item-Item)'
    best_rmse = item_based_rmse

print(f"\n🏆 BEST MODEL: {best_model}")
print(f"   RMSE: {best_rmse:.4f}")

In [ ]:
print("\n" + "="*80)
print("SAVING RECOMMENDATION SYSTEM (FULL DATA)")
print("="*80)

import os
os.makedirs('models/recommendation', exist_ok=True)

# Save complete system
recommendation_system = {
    'user_item_matrix': user_item_matrix,
    'user_similarity_matrix': user_similarity,
    'item_similarity_matrix': item_similarity,
    'user_means': user_means,
    'user_based_rmse': user_based_rmse,
    'item_based_rmse': item_based_rmse,
    'user_based_mae': user_based_mae,
    'item_based_mae': item_based_mae,
    'best_model': best_model,
    'n_users': n_users,
    'n_products': n_products,
    'n_ratings': n_ratings,
    'sparsity': sparsity,
    'similarity_type': 'adjusted_cosine'
}

with open('models/recommendation/full_data_recommendation_system.pkl', 'wb') as f:
    pickle.dump(recommendation_system, f)
print("✓ Full recommendation system saved")

# Save individual components
with open('models/recommendation/user_item_matrix_full.pkl', 'wb') as f:
    pickle.dump(user_item_matrix, f)
with open('models/recommendation/user_similarity_full.pkl', 'wb') as f:
    pickle.dump(user_similarity, f)
with open('models/recommendation/item_similarity_full.pkl', 'wb') as f:
    pickle.dump(item_similarity, f)
with open('models/recommendation/user_means_full.pkl', 'wb') as f:
    pickle.dump(user_means, f)
print("✓ Individual components saved")